In [2]:
%store -r generated_full_test_list

In [3]:
%store -r test_paired_summaries
#dict with Key is scene and Value as tupple (text, summary, human-made-flag)

In [6]:
#Looking up scene by number
print(list(test_paired_summaries.keys())[10]) #Julius_Caesar-Act IV-Scene I
print(list(test_paired_summaries.keys())[23]) #MacBeth-Act I-Scene VI
print(list(test_paired_summaries.keys())[60]) #Twelfth_Night-Act IV-Scene I
print(list(test_paired_summaries.keys())[83]) #Sonnet 40

Julius_Caesar-Act IV-Scene I
MacBeth-Act I-Scene VI
Twelfth_Night-Act IV-Scene I
Sonnet 40


In [7]:
#Looking at tuned T5 genertaed results
print(generated_full_test_list[10]) #Julius_Caesar-Act IV-Scene I
print(generated_full_test_list[23]) #MacBeth-Act I-Scene VI
print(generated_full_test_list[60]) #Twelfth_Night-Act IV-Scene I
print(generated_full_test_list[83]) #Sonnet 40


In Caesar’s funeral, Cinna, a poet, and Cinna, the poet, are arrested and charged with conspiracy. The poet, Cinna, tries to convince him of Caesar’s death.
Donalbain, Banquo, Lennox, Ross, Angus and Macduff attend the castle. Banquo and Lennox reaffirm their admiration for Macbeth’s love. The
Sebastian tries to convince Clown that he is not sent for him. Clown tries to convince Sebastian that he is not sent for him. Sebastian reassures him that he is not sent for him. Cl
Love understands it is a greater grief to bear love's wrong, than hate's known injury. Love understands it is a greater grief to bear love's wrong, than hate's known injury. Love


In [8]:
#Looking at human results
print(list(test_paired_summaries.values())[10][1]) #Julius_Caesar-Act IV-Scene I
print(list(test_paired_summaries.values())[23][1]) #MacBeth-Act I-Scene VI
print(list(test_paired_summaries.values())[60][1]) #Twelfth_Night-Act IV-Scene I
print(list(test_paired_summaries.values())[83][1]) #Sonnet 40

Enter ANTONY, OCTAVIUS, and LEPIDUS. (4.1.1)—Antony, Octavius, and Lepidus agree on a list of those who will be executed in a purge of suspected enemies. Antony, wanting to find a way to reduce the expenses of Caesar's legacies, sends Lepidus for Caesar's will. Exit LEPIDUS. (4.1.12)—Antony tells Octavius that Lepidus is a donkey who should be discarded as soon as they have no more use for him. Antony and Octavius agree that they need to take immediate action against the armies of Brutus and Cassius.
King Duncan arrives at Macbeth's castle and is greeted by Lady Macbeth.
Enter SEBASTIAN and Clown (4.1.1) — The Clown has been sent by Olivia to get Cesario to come and speak with her, but Sebastian (who looks exactly like his twin, Viola, in her disguise as Cesario) won't admit that he is Cesario. The Clown is sarcastic with Sebastian; Sebastian is annoyed with the Clown. Enter SIR ANDREW, SIR TOBY BELCH, and FABIAN (4.1.24) — Sir Andrew, thinking he has caught up with the cowardly Cesari

In [9]:
from rouge_score import rouge_scorer
from bert_score import score as bertscore

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

import numpy as np

from bert_score import score as bertscore

import torch

from transformers import AutoModel


/Users/chadadelman/anaconda3/envs/chad_env/lib/python3.11/site-packages/pandas/core/arrays/masked.py:60: UserWarning: Pandas requires version '1.3.6' or newer of 'bottleneck' (version '1.3.5' currently installed).
  from pandas.core import (


In [10]:
#similarity methods functions (made with help of ChatGPT)
def compute_rouge(text1, text2):
    """
    Compute ROUGE between two texts.
    Returns F1 scores for ROUGE-1, ROUGE-2, ROUGE-L.
    """
    scorer = rouge_scorer.RougeScorer(
        ['rouge1', 'rouge2', 'rougeL'],
        use_stemmer=True
    )
    # Note: scorer.score(reference, prediction)
    scores = scorer.score(text2, text1)

    return {
        "rouge1": scores["rouge1"].fmeasure,
        "rouge2": scores["rouge2"].fmeasure,
        "rougeL": scores["rougeL"].fmeasure,
    }


def compute_cosine_similarity(text1, text2):
    """
    Compute cosine similarity between two texts using TF-IDF vectors.
    This avoids loading any transformer model and is stable in most environments.
    """
    vectorizer = TfidfVectorizer()
    tfidf_matrix = vectorizer.fit_transform([text1, text2])  # shape (2, vocab_size)

    cos_sim = cosine_similarity(tfidf_matrix[0:1], tfidf_matrix[1:2])  # (1,1)
    return float(cos_sim[0, 0])

def compute_bertscore(text1: str, text2: str):
    """
    Compute BERTScore (Precision, Recall, F1) between two texts.
    text1 = candidate
    text2 = reference
    """
    P, R, F1 = bertscore(
        cands=[text1],
        refs=[text2],
        lang="en"
    )
    return {
        "precision": float(P[0]),
        "recall": float(R[0]),
        "f1": float(F1[0]),
    }


In [12]:
rouge_score_list_2_1_all_test=[]
cosine_score_list_2_1_all_test=[]
bertscore_list_2_1_all_test=[]

#Scoring average results vs the human anwers in all metrics 
for i in range(len(generated_full_test_list)):
    t5_summary = generated_full_test_list[i]
    human_summary = list(test_paired_summaries.values())[i][1]
    
    rouge_score_list_2_1_all_test.append(compute_rouge(t5_summary, human_summary))
    cosine_score_list_2_1_all_test.append(compute_cosine_similarity(t5_summary, human_summary))
    bertscore_list_2_1_all_test.append(compute_bertscore(t5_summary, human_summary))
    
    if i%10==0:
        print(i)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


0


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

10


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

...

120


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

130


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['pooler.dense.bias', 'pooler.dense.weight']
You sho

140


In [13]:
%store rouge_score_list_2_1_all_test
%store cosine_score_list_2_1_all_test
%store bertscore_list_2_1_all_test

Stored 'rouge_score_list_2_1_all_test' (list)
Stored 'cosine_score_list_2_1_all_test' (list)
Stored 'bertscore_list_2_1_all_test' (list)


In [14]:
#Looking at data
print(rouge_score_list_2_1_all_test[0])
print(bertscore_list_2_1_all_test[0])

{'rouge1': 0.22641509433962265, 'rouge2': 0.0784313725490196, 'rougeL': 0.1509433962264151}
{'precision': 0.8708274960517883, 'recall': 0.8535978198051453, 'f1': 0.8621265888214111}


In [15]:
#Compute averages across all testi dataset
rouge_score_list_2_1_all_test_avg = [0,0,0]
cosine_score_list_2_1_all_test_avg = 0
bertscore_list_2_1_all_test_avg = [0,0,0]

passage_count=len(rouge_score_list_2_1_all_test)

for i in range(passage_count):
    rouge_score_list_2_1_all_test_avg[0] += rouge_score_list_2_1_all_test[i]["rouge1"]
    rouge_score_list_2_1_all_test_avg[1] += rouge_score_list_2_1_all_test[i]["rouge2"]
    rouge_score_list_2_1_all_test_avg[2] += rouge_score_list_2_1_all_test[i]["rougeL"]
    cosine_score_list_2_1_all_test_avg += cosine_score_list_2_1_all_test[i]
    bertscore_list_2_1_all_test_avg[0] += bertscore_list_2_1_all_test[i]["precision"]
    bertscore_list_2_1_all_test_avg[1] += bertscore_list_2_1_all_test[i]["recall"]
    bertscore_list_2_1_all_test_avg[2] += bertscore_list_2_1_all_test[i]["f1"]

rouge_score_list_2_1_all_test_avg[0] /=passage_count
rouge_score_list_2_1_all_test_avg[1] /=passage_count
rouge_score_list_2_1_all_test_avg[2] /=passage_count
cosine_score_list_2_1_all_test_avg /=passage_count
bertscore_list_2_1_all_test_avg[0] /=passage_count
bertscore_list_2_1_all_test_avg[1] /=passage_count
bertscore_list_2_1_all_test_avg[2] /=passage_count




In [16]:
print(rouge_score_list_2_1_all_test_avg)
print(cosine_score_list_2_1_all_test_avg)
print(bertscore_list_2_1_all_test_avg)

[0.19201016996747672, 0.023866626356047818, 0.14086413721102067]
0.20359771457502923
[0.8350805914993827, 0.8300009796805415, 0.832273763545016]
